## SETUP LOGFIRE

In [1]:
import os, time, warnings
warnings.filterwarnings("ignore")


import logfire 
from dotenv import load_dotenv

load_dotenv()


# Verify keys
print("LOGFIRE_TOKEN  :", "✅" if os.getenv("LOGFIRE_TOKEN")  else "❌  missing")
print("GROQ_API_KEY   :", "✅" if os.getenv("GROQ_API_KEY")   else "❌  missing")
print("GEMINI_API_KEY :", "✅" if os.getenv("GEMINI_API_KEY") else "❌  missing")

LOGFIRE_TOKEN  : ✅
GROQ_API_KEY   : ✅
GEMINI_API_KEY : ✅


In [2]:
import logfire

logfire.configure()
logfire.info('hello , {place}!', place = 'world')

Logfire project URL: https://logfire-us.pydantic.dev/sahilsawant7120/logfire-demo

17:48:56.003 hello , world!


## Simple INFO

In [3]:
logfire.info("notebook_started",
              notebook_name = "pydantic_logfire.ipynb",
              timestamp = time.time(),
              part = "part_1",
              student = "Sahil Sawant",
              tool = "Pydantic Logfire"
              )

17:48:56.014 notebook_started


## Implementing TRACE

In [4]:
with logfire.span("data_processing_simulation", dataset="llm_course", rows=1000):
    logfire.info("step_started", step=1, action="loading data")
    time.sleep(0.3)

    logfire.info("step_started", step=2, action="transforming", columns=12)
    time.sleep(0.2)

    logfire.info("step_started", step=3, action="saving results", output="/tmp/out.csv")

17:48:56.027 data_processing_simulation
17:48:56.028   step_started
17:48:56.329   step_started
17:48:56.532   step_started


## Experiment 2 — Structured Logging with Pydantic Models

In [5]:
from pydantic import BaseModel
from typing import Optional

class LLMRequest(BaseModel):
    user_id: str
    session_id: str
    query: str
    model : str
    temperature: float = 0.7
    max_tokens: Optional[int] = None

class LLMResponse(BaseModel):
    answer: str
    input_tokens: int
    output_tokens: int
    latency_ms : float
    model_used: str

In [6]:
#Passing DATA

request = LLMRequest(
    user_id="user_123",
    session_id="session_456",
    query="What is the capital of France?",
    model="gpt-4",
    temperature=0.5,
    max_tokens=100
)

with logfire.span("llm_request",
                user_id=request.user_id,
                session_id=request.session_id,
                model_used=request.model 
                ):

    logfire.info("request_received" , **request.model_dump())

    time.sleep(0.1)

    response = LLMResponse(
        answer="The capital of France is Paris.",
        input_tokens=10,
        output_tokens=7,
        latency_ms=120.5,
        model_used="llm-4"
    )
    logfire.info("response_generated" , **response.model_dump())

print(response)


17:48:56.632 llm_request
17:48:56.633   request_received
17:48:56.734   response_generated
answer='The capital of France is Paris.' input_tokens=10 output_tokens=7 latency_ms=120.5 model_used='llm-4'


## Experiment 3 - instrumen t Groq 

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage


logfire.instrument_openai()

llm_groq = ChatOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
    model_name="openai/gpt-oss-20b",
    temperature=0.4,
)

#make an groq request

print("Calling Groq LLM.....")
response = llm_groq.invoke([
    HumanMessage(content="Explain me about the benefits of using Groq for LLM inference in exactly 2 sentences.")
])

print(response.content)

Calling Groq LLM.....
17:49:00.276 Chat Completion with 'openai/gpt-oss-20b' [LLM]
Groq's tensor processing architecture delivers ultra‑low latency and high throughput for LLM inference, enabling real‑time applications with minimal response times. Its energy‑efficient design and cost‑effective scaling reduce operational expenses while maintaining competitive performance against GPU and TPU alternatives.


## Experiment 4 Instrument gemini

In [8]:
llm_gemini = ChatOpenAI(
    base_url = "https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key = os.getenv("GEMINI_API_KEY"),
    model = "gemini-2.5-flash-lite",
    temperature = 0.4,
)

print("Calling Gemini (gemini-2.5-flash-lite)...")
try:
    response = llm_gemini.invoke([
        HumanMessage(content="Explain what an observability 'span' is, in exactly 2 sentences.")
    ])
    print(f"\n🔵 Gemini Response:\n{response.content}")
    print("\n✅ In Logfire dashboard:")
    print("  → You now see BOTH 'openai/gpt-oss-20b' and 'gemini-2.5-flash-lite' in traces")
    print("  → Same query, different providers - compare latency and token usage")
except Exception as e:
    print(f"⚠️ Gemini call failed: {e}")
    print("    Check your GEMINI_API_KEY in .env")

Calling Gemini (gemini-2.5-flash-lite)...
17:56:48.510 Chat Completion with 'gemini-2.5-flash-lite' [LLM]

🔵 Gemini Response:
An observability span represents a single, timed operation within a distributed system, capturing its start time, duration, and any associated metadata. It's a fundamental building block for tracing, allowing you to visualize the flow of requests and diagnose performance bottlenecks across multiple services.

✅ In Logfire dashboard:
  → You now see BOTH 'openai/gpt-oss-20b' and 'gemini-2.5-flash-lite' in traces
  → Same query, different providers - compare latency and token usage


## Experiment 5 - Gemini vs Groq side by side waterfall trace

In [9]:
query = "What is the difference between RAG and fine-tuning? Give answer in5 bullet points."

with logfire.span("model_comparison", query=query, num_models=2):
    
    # --- Groq ---
    with logfire.span("groq_call", model="openai/gpt-oss-20b", provider="groq"):
        t0 = time.time()
        r_groq = llm_groq.invoke([HumanMessage(content=query)])
        groq_ms = round((time.time() - t0) * 1000, 1)
        logfire.info("groq_done", latency_ms=groq_ms, answer_len=len(r_groq.content))
        
    # --- Gemini ---
    with logfire.span("gemini_call", model="gemini-2.5-flash-lite", provider="google"):
        t0 = time.time()
        try:
            r_gemini = llm_gemini.invoke([HumanMessage(content=query)])
            gemini_ms = round((time.time() - t0) * 1000, 1)
            logfire.info("gemini_done", latency_ms=gemini_ms, answer_len=len(r_gemini.content))
            gemini_answer = r_gemini.content
        except Exception as e:
            logfire.warning("gemini_failed", error=str(e))

print(f" Groq ({groq_ms} ms):\n{r_groq.content}\n")
print(f" Gemini ({gemini_ms} ms):\n{gemini_answer}\n")

18:05:53.607 model_comparison
18:05:53.607   groq_call
18:05:53.609     Chat Completion with 'openai/gpt-oss-20b' [LLM]
18:05:54.368     groq_done
18:05:54.369   gemini_call
18:05:54.371     Chat Completion with 'gemini-2.5-flash-lite' [LLM]
18:05:56.444     gemini_done
 Groq (759.7 ms):
- **Training vs. Inference**  
  - *Fine‑tuning* modifies the model’s weights during training on a new dataset.  
  - *RAG* keeps the base model unchanged; it augments generation at inference by retrieving relevant documents.

- **Data Usage**  
  - Fine‑tuning consumes a large, curated training set to embed knowledge into the model.  
  - RAG pulls up‑to‑date information from an external index (e.g., a vector database) on the fly.

- **Adaptability & Freshness**  
  - Fine‑tuned models are static; updating them requires re‑training.  
  - RAG can instantly reflect new documents or knowledge without retraining.

- **Model Size & Complexity**  
  - Fine‑tuning often requires a larger, more specialized m

## TRACED RAG PIPELINE

In [10]:
import json
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

#load json documents

with open("documents.json") as f:
    raw_docs = json.load(f)

#convert docs in langchain compatible documents

DOCS = [
    Document(page_content=d["content"], metadata={"topic": d["topic"], "source": d["source"]})
    for d in raw_docs
]
print(f"Loaded {len(DOCS)} documents: {[d.metadata['topic'] for d in DOCS]}")


Loaded 6 documents: ['RAG', 'Guardrails', 'Gateway', 'Observability', 'Evals', 'Fine-tuning']


In [11]:
DOCS

[Document(metadata={'topic': 'RAG', 'source': 'doc_1'}, page_content='Retrieval-Augmented Generation (RAG) combines information retrieval with text generation. When a user asks a question, RAG first retrieves relevant documents from a knowledge base using vector similarity search, then passes those documents along with the question to an LLM. This grounds the answer in actual content, which significantly reduces hallucinations compared to pure LLM generation.'),
 Document(metadata={'topic': 'Guardrails', 'source': 'doc_2'}, page_content='LLM Guardrails are safety controls that sit between the user and the language model. They run before the LLM sees the input (input rails) and after the LLM generates output (output rails). NVIDIA NeMo Guardrails uses a domain-specific language called Colang to define rules declaratively. Common guardrails include prompt injection detection, PII filtering, toxicity filtering, and topic restriction.'),
 Document(metadata={'topic': 'Gateway', 'source': 'd

In [12]:
## embeddings

enbeddings = GoogleGenerativeAIEmbeddings(
    model = "gemini-embedding-2-preview",
    google_api_key = os.getenv("GEMINI_API_KEY")
)

#implementing faiss vector store

vector_store = FAISS.from_documents(DOCS, enbeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
print(f"Retriever ready. Top 2 docs will be returned for each query.")

Retriever ready. Top 2 docs will be returned for each query.


## Data Retrival pipeline

In [13]:
# -- RAG with tracing --
def rag(question: str, user_id: str = "anonymous") -> str:
    with logfire.span("rag_pipeline", question=question, user_id=user_id):
        docs = retriever.invoke(question)
        logfire.info("docs_retrieved",
                     topics=[d.metadata["topic"] for d in docs],
                     num_docs=len(docs))
        context = "\n\n".join(
            f"[{d.metadata['topic']}] {d.page_content}" for d in docs
        )
        prompt = (
            f"Answer the question based only on the context below.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {question}\n\nAnswer concisely:"
        )
        return llm_groq.invoke(prompt).content

In [14]:
answer = rag("How does a RAG reduce hallucination", user_id="student_001")
print(f"RAG Answer:\n{answer}")

18:44:37.319 rag_pipeline
18:44:37.887   docs_retrieved
18:44:37.889   Chat Completion with 'openai/gpt-oss-20b' [LLM]
RAG Answer:
By grounding the LLM’s output in retrieved documents, RAG limits the model to facts it can verify, so it produces fewer hallucinations.


## REACT AGENT

In [15]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage, AIMessage



In [16]:
#Craeting retriver as a tool for agent
@tool
def search_knowledge_base(query : str) -> str:
    """Searches the knowledge base for relevant information."""
    docs = vector_store.similarity_search(query, k=2)
    return "\n\n".join(
        f"[{d.metadata['topic']}] {d.page_content}" for d in docs
    )

In [17]:
agent = create_agent(
    model=llm_groq,
    tools=[search_knowledge_base],
    system_prompt = (
        "You are an AI assistant that can answer questions based on a knowledge base. "
        "Use the provided tools to search the knowledge base and provide accurate answers."
        "if answer is not found in the knowledge base, say 'I don't know' and do not make up an answer."
    )
) 

In [18]:
# - Traced agent runner
def run_agent(question: str, user_id: str = "anonymous"):
    with logfire.span("agent_run", question=question, user_id=user_id):
        result = agent.invoke({"messages": [HumanMessage(content=question)]})
        
        # Last message may be a ToolMessage or an AIMessage with empty content
        # - iterate backwards to find the last AIMessage with actual text
        last_ai = next(
            (m for m in reversed(result["messages"]) if isinstance(m, AIMessage)),
            None
        )
        answer    = last_ai.content if last_ai else ""
        used_tool = any(isinstance(m, ToolMessage) for m in result["messages"])
        
        logfire.info("agent_done", used_tool=used_tool, answer_length=len(answer))
        return answer, used_tool

In [19]:
## testing with queries
queries = [
    ("What is the capital of France?", "sarvesh"),
    ("What is LLm observability and which tools provide it?", "sahil"),
    ("How do LLm Guardrails work?", "bhushan"),
    ("How many continents are there?", "shivam"),
    ("What is the chemical symbol for gold?", "rohan")
]

In [20]:
for q, uid in queries:
    print(f"{'='*55}")
    answer, used_tool = run_agent(q, user_id=uid)
    print(f"Q: {q}")
    print(f"Tool used: {used_tool}   {'<-- retrieved from KB' if used_tool else '<-- answered directly'}")
    print(f"A: {answer[:300]}")

19:38:14.580 agent_run
19:38:14.596   Chat Completion with 'openai/gpt-oss-20b' [LLM]
19:38:15.767   Chat Completion with 'openai/gpt-oss-20b' [LLM]
19:38:16.557   agent_done
Q: What is the capital of France?
Tool used: True   <-- retrieved from KB
A: I don't know.
19:38:16.558 agent_run
19:38:16.563   Chat Completion with 'openai/gpt-oss-20b' [LLM]
19:38:17.521   Chat Completion with 'openai/gpt-oss-20b' [LLM]
19:38:18.399   agent_done
Q: What is LLm observability and which tools provide it?
Tool used: True   <-- retrieved from KB
A: **LLM Observability**  
LLM observability is the practice of monitoring, tracing, and debugging language‑model applications while they run in production. It typically includes:

- **Token usage & cost tracking** per request  
- **Latency measurement** across all pipeline stages (prompt, inference, p
19:38:18.401 agent_run
19:38:18.404   Chat Completion with 'openai/gpt-oss-20b' [LLM]
19:38:19.424   Chat Completion with 'openai/gpt-oss-20b' [LLM]
19:38:21.